In [ ]:
import os

# path setting
current = os.getcwd()
while os.path.basename(current) != "Data_center_and_fossil_energy_Replication":
    current = os.path.dirname(current)

BASE_PATH = current

RAW = os.path.join(BASE_PATH, 'Data', 'raw')
TEMP = os.path.join(BASE_PATH, 'Data', 'temp')
USE = os.path.join(BASE_PATH, 'Data', 'use')
FIGURES = os.path.join(BASE_PATH, 'Results', 'Figures')
TABLES = os.path.join(BASE_PATH, 'Results', 'Tables')

for path in [RAW, TEMP, USE, FIGURES, TABLES]:
    os.makedirs(path, exist_ok=True)


In [ ]:
"""
Calculate AI center exposure within coal plant buffers
(inverse-distance weighting + multiple time windows version - 15km/25km/50km)
- Input 1: gem_coal_plants_multi_record_sa_sinceoperating.dta (coal plant multi-record survival data)
- Input 2: sp_global_ai_center.xlsx (AI center data)
- Input 3: gadm_410.gpkg (GADM administrative boundary data) 🆕
- Output: gem_coal_plants_multi_record_sa_proximity_exposure_15_25_50km_periods.dta
- Core metric: Proximity Exposure (PE) = Σ(1/distance_10km)
- Time windows: before06, 06-15, 16-19, 20-24
- Buffer radii: 15km, 25km, 50km
"""

import os
import pandas as pd
import numpy as np
from shapely.geometry import Point
import geopandas as gpd
from tqdm import tqdm
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

def calculate_distance_vectorized(lat1, lon1, lat2_array, lon2_array):
    """
    Vectorized distance calculation between points (Haversine formula)
    
    Parameters:
        lat1, lon1: latitude and longitude of one point
        lat2_array, lon2_array: arrays of latitudes and longitudes of multiple points
    
    Returns:
        Distance array (unit: km)
    """
    # Convert to radians
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2_array)
    lon2_rad = np.radians(lon2_array)
    
    # Haversine formula
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # Earth radius (km)
    r = 6371
    
    return c * r

def match_gadm_attributes(df_coal, gadm_path):
    """
    🆕 Match GADM administrative attributes by plant coordinates
    
    Parameters:
        df_coal: DataFrame (with Latitude and Longitude)
        gadm_path: path to GADM gpkg file
    
    Returns:
        Matched DataFrame
    """
    print("\n🗺️  Matching GADM administrative attributes...")
    
    # Read GADM data
    try:
        gadm_gdf = gpd.read_file(gadm_path)
        print(f"  ✓ GADM data loaded: {len(gadm_gdf)} records")
        print(f"  ✓ GADM fields: {list(gadm_gdf.columns)}")
    except Exception as e:
        print(f"  ❌ Failed to read GADM: {e}")
        return df_coal
    
    # Create GeoDataFrame using plant coordinates
    geometry = [Point(xy) for xy in zip(df_coal['Longitude'], df_coal['Latitude'])]
    coal_gdf = gpd.GeoDataFrame(df_coal, geometry=geometry, crs='EPSG:4326')
    
    # Ensure CRS consistency
    if gadm_gdf.crs != coal_gdf.crs:
        print(f"  - Convert GADM CRS: {gadm_gdf.crs} → {coal_gdf.crs}")
        gadm_gdf = gadm_gdf.to_crs(coal_gdf.crs)
    
    # Spatial join
    print("  - Running spatial join (may take a few minutes)...")
    start_time = time.time()
    
    coal_matched = gpd.sjoin(coal_gdf, gadm_gdf, how='left', predicate='within')
    
    elapsed = time.time() - start_time
    print(f"  ✓ Spatial join done, time: {elapsed/60:.1f} min")
    
    # Match summary
    matched_count = coal_matched['index_right'].notna().sum()
    print(f"  ✓ Matched: {matched_count:,}/{len(df_coal):,} records ({matched_count/len(df_coal)*100:.1f}%)")
    
    # Get GADM fields (exclude geometry and index_right)
    gadm_cols = [col for col in coal_matched.columns 
                 if col not in df_coal.columns 
                 and col not in ['geometry', 'index_right']]
    
    if gadm_cols:
        print(f"  ✓ Matched GADM fields ({len(gadm_cols)}): {gadm_cols[:10]}")
        if len(gadm_cols) > 10:
            print(f"    ... {len(gadm_cols)-10} more fields")
    
    # Drop extra columns from spatial join
    cols_to_drop = ['geometry', 'index_right']
    coal_matched = coal_matched.drop(columns=[c for c in cols_to_drop if c in coal_matched.columns])
    
    # Convert back to regular DataFrame
    coal_matched = pd.DataFrame(coal_matched)
    
    return coal_matched

def process_proximity_exposure():
    """
    Main function: calculate inverse-distance-based exposure metrics
    (multiple time windows)
    """
    
    # ===== 1. Read data =====
    coal_plants_path = os.path.join(TEMP, "gem_coal_plants_multi_record_sa_sinceoperating.dta")
    df_coal = pd.read_stata(coal_plants_path)
    
    ai_center_path = os.path.join(RAW, "SPGlobal_Export.xlsx")
    df_ai = pd.read_excel(ai_center_path, sheet_name='Sheet1')
    
    gadm_path = os.path.join(RAW, "gadm_410.gpkg")  # 🆕
    
    # ===== 1.5 Match GADM attributes 🆕 =====
    df_coal = match_gadm_attributes(df_coal, gadm_path)
    
    # ===== 2. Data preprocessing =====
    print("\n🔧 Step 2/5: Data preprocessing...")
    
    # Process AI center built year
    df_ai['YR_BUILT'] = pd.to_numeric(df_ai['YR_BUILT'], errors='coerce')
    
    # Drop missing values
    df_ai_filtered = df_ai.dropna(subset=['LATITUDE', 'LONGITUDE', 'YR_BUILT']).copy()
    print(f"  ✓ Valid AI centers: {len(df_ai_filtered):,}")
    
    df_coal = df_coal.dropna(subset=['Latitude', 'Longitude', 'year'])
    print(f"  ✓ Valid coal plant records: {len(df_coal):,}")
    
    # Convert to numpy arrays
    ai_lat = df_ai_filtered['LATITUDE'].values
    ai_lon = df_ai_filtered['LONGITUDE'].values
    ai_year = df_ai_filtered['YR_BUILT'].values.astype(int)
    
    print(f"  ✓ AI center year range: {ai_year.min()} - {ai_year.max()}")
    
    # Show year distribution
    year_counts = pd.Series(ai_year).value_counts().sort_index()
    print(f"  ✓ AI center year distribution (first 10 years):")
    for year, count in year_counts.head(10).items():
        print(f"    - {year}: {count}")
    if len(year_counts) > 10:
        print(f"    ... ({len(year_counts)} years total)")
    
    # ===== 3. Define parameters =====
    print("\n📊 Step 3/5: Define exposure parameters...")
    
    buffer_distances = [15, 25, 50]  # 15km, 25km, 50km (unit: km)
    
    # Modified: four time windows
    time_windows = {
        'before06': (None, 2005),      # 2005 and earlier
        '06_15': (2006, 2015),          # 2006-2015
        '16_19': (2016, 2019),          # 2016-2019
        '20_24': (2020, 2024)           # 2020-2024
    }
    
    print(f"  ✓ Buffer radii: {[f'{d}km' for d in buffer_distances]}")
    print(f"  ✓ Time windows:")
    for window_name, (start, end) in time_windows.items():
        if start is None:
            print(f"    • {window_name}: ≤{end}")
        else:
            print(f"    • {window_name}: {start}-{end}")
    print(f"  ✓ Total variables: {len(buffer_distances)} × {len(time_windows)} = {len(buffer_distances) * len(time_windows)}")
    
    # ===== 4. Create new variables =====
    print("\n🎯 Step 4/5: Create exposure variables...")
    
    new_columns = []
    
    for distance_km in buffer_distances:
        for window_name in time_windows.keys():
            # Variable naming: ai_proximity_[radius]km_[time window]
            col_name = f'ai_proximity_{distance_km}km_{window_name}'
            df_coal[col_name] = 0.0  # Initialize as float
            new_columns.append(col_name)
    
    print(f"  ✓ Created {len(new_columns)} exposure variables:")
    for i, col in enumerate(new_columns, 1):
        print(f"    {i:2d}. {col}")
    
    # ===== 5. Calculate exposure metrics (core calculation) =====
    print("\n🚀 Step 5/5: Calculate inverse-distance-weighted exposure metrics...")
    print(f"  - Formula: PE = Σ(1/distance_km) for AI centers built in the time window within the buffer")
    print(f"  - Distance unit: km")
    print(f"  - Time window notes:")
    for window_name, (start, end) in time_windows.items():
        if start is None:
            print(f"    • {window_name}: only AI centers built in or before {end}")
        else:
            print(f"    • {window_name}: only AI centers built in {start}-{end}")
    print(f"  - Total calculations: {len(df_coal):,} records")
    
    start_time = time.time()
    processed_count = 0
    last_report_time = start_time
    
    # Process by GEM_unit_phase_ID group
    grouped = df_coal.groupby('GEM_unit_phase_ID')
    
    for gem_id, group_df in tqdm(grouped, desc="Processing coal plants", ncols=80):
        # Get coordinates of this coal plant
        coal_lat = group_df['Latitude'].iloc[0]
        coal_lon = group_df['Longitude'].iloc[0]
        
        # Vectorized calculation of distances from this coal plant to all AI centers (km)
        distances_km = calculate_distance_vectorized(coal_lat, coal_lon, ai_lat, ai_lon)
        
        # Calculate exposure metrics for each year of this coal plant
        for idx, row in group_df.iterrows():
            current_year = int(row['year'])
            
            # For each buffer radius
            for distance_km in buffer_distances:
                
                # Find AI centers within the buffer
                within_buffer = distances_km <= distance_km
                
                # For each time window
                for window_name, (start_year, end_year) in time_windows.items():
                    col_name = f'ai_proximity_{distance_km}km_{window_name}'
                    
                    # Filter conditions:
                    # 1. Within the buffer
                    # 2. Built within the time window
                    # 3. Built year <= current year (already built)
                    if start_year is None:
                        # before06: built in or before 2005
                        valid_ai = within_buffer & (ai_year <= end_year) & (ai_year <= current_year)
                    else:
                        # Other windows: built within the window
                        valid_ai = within_buffer & (ai_year >= start_year) & (ai_year <= end_year) & (ai_year <= current_year)
                    
                    if valid_ai.sum() > 0:
                        # Calculate sum of inverse distances (distance unit: km)
                        valid_distances_km = distances_km[valid_ai]
                        # Avoid division by zero: if distance < 0.1km (100m), set to 0.1km
                        valid_distances_km = np.maximum(valid_distances_km, 0.1)
                        valid_distances_10km = valid_distances_km / 10.0
                        # PE = Σ(1/distance_10km)
                        proximity_exposure = np.sum(1.0 / valid_distances_10km)
                        
                        df_coal.at[idx, col_name] = proximity_exposure
                    else:
                        df_coal.at[idx, col_name] = 0.0
            
            processed_count += 1
        
        # Progress report
        current_time = time.time()
        if processed_count % 1000 == 0 or (current_time - last_report_time) > 60:
            elapsed = current_time - start_time
            speed = processed_count / elapsed if elapsed > 0 else 0
            remaining = len(df_coal) - processed_count
            eta_seconds = remaining / speed if speed > 0 else 0
            
            print(f"\n{'='*80}")
            print(f"Progress report [{datetime.now().strftime('%H:%M:%S')}]")
            print(f"{'='*80}")
            print(f"Processed: {processed_count:,}/{len(df_coal):,} records ({processed_count/len(df_coal)*100:.1f}%)")
            print(f"Speed: {speed:.2f} records/sec")
            print(f"Elapsed: {elapsed/60:.1f} min")
            print(f"Remaining: {eta_seconds/60:.1f} min")
            print(f"ETA: {(datetime.now() + pd.Timedelta(seconds=eta_seconds)).strftime('%H:%M:%S')}")
            print(f"{'='*80}\n")
            
            last_report_time = current_time
    
    total_elapsed = time.time() - start_time
    print(f"\n✓ Done! Total time: {total_elapsed/60:.1f} min")
    
    # ===== 6. Data validation and statistics =====
    print("\n📈 Exposure statistics:")
    print("="*80)
    
    for col in new_columns:
        col_data = df_coal[col]
        
        print(f"\n{col}:")
        print(f"  Basic stats:")
        print(f"    - Mean: {col_data.mean():.6f}")
        print(f"    - Median: {col_data.median():.6f}")
        print(f"    - Std: {col_data.std():.6f}")
        print(f"    - Min: {col_data.min():.6f}")
        print(f"    - Max: {col_data.max():.6f}")
        print(f"  Distribution:")
        print(f"    - Non-zero records: {(col_data > 0).sum():,} ({(col_data > 0).sum()/len(df_coal)*100:.1f}%)")
        print(f"    - P25: {col_data.quantile(0.25):.6f}")
        print(f"    - P75: {col_data.quantile(0.75):.6f}")
        print(f"    - P90: {col_data.quantile(0.90):.6f}")
        print(f"    - P99: {col_data.quantile(0.99):.6f}")
    
    # ===== 7. Time window comparison =====
    print("\n📊 Time window comparison:")
    print("="*80)
    
    for distance_km in buffer_distances:
        print(f"\n{distance_km}km buffer - comparison across time windows:")
        
        comparison_data = []
        for window_name in time_windows.keys():
            col_name = f'ai_proximity_{distance_km}km_{window_name}'
            col_data = df_coal[col_name]
            
            comparison_data.append({
                'Time window': window_name,
                'Mean': f"{col_data.mean():.6f}",
                'Median': f"{col_data.median():.6f}",
                'Max': f"{col_data.max():.6f}",
                'Non-zero share': f"{(col_data > 0).sum()/len(df_coal)*100:.1f}%"
            })
        
        comparison_df = pd.DataFrame(comparison_data)
        print(comparison_df.to_string(index=False))
    
    # ===== 8. Sample data preview =====
    print("\n📋 Sample data preview:")
    print("="*80)
    
    sample_cols = ['GEM_unit_phase_ID', 'year'] + new_columns[:6]
    print(df_coal[sample_cols].head(10).to_string())
    
    # ===== 9. Save results =====
    output_path = os.path.join(TEMP, "gem_coal_plants_multi_record_sa_proximity_exposure_15_25_50km_periods.dta")
    df_coal.to_stata(output_path, write_index=False, version=118)
    
    print(f"\n✅ Results saved to: {output_path}")
    
    return df_coal

def quick_check_results():
    """
    Quick check of the output file
    """
    output_path = os.path.join(TEMP, "gem_coal_plants_multi_record_sa_proximity_exposure_15_25_50km_periods.dta")
    
    try:
        df = pd.read_stata(output_path)
        
        print("\n🔍 Output file check:")
        print("="*80)
        
        # Check exposure variables
        proximity_vars = [col for col in df.columns if col.startswith('ai_proximity_')]
        
        # Check GADM variables 🆕
        gadm_vars = [col for col in df.columns if col.startswith('GID_') or col.startswith('NAME_') or col.startswith('COUNTRY')]
        
        print(f"✓ Total records: {len(df):,}")
        print(f"✓ Exposure variables: {len(proximity_vars)}")
        print(f"✓ GADM variables: {len(gadm_vars)} 🆕")
        
        if gadm_vars:
            print(f"✓ GADM variable list:")
            for var in gadm_vars[:10]:
                print(f"  - {var}")
            if len(gadm_vars) > 10:
                print(f"  ... {len(gadm_vars)-10} more variables")
        
        print(f"\n✓ Exposure variable list:")
        for var in proximity_vars:
            print(f"  - {var}")
        
        # Show by distance group
        for distance_km in [15, 25, 50]:
            distance_vars = [v for v in proximity_vars if f'_{distance_km}km_' in v]
            print(f"\n{distance_km}km buffer variable stats:")
            for var in distance_vars:
                mean_val = df[var].mean()
                max_val = df[var].max()
                nonzero = (df[var] > 0).sum()
                print(f"  {var}: mean={mean_val:.6f}, max={max_val:.6f}, non-zero={nonzero:,}")
        
    except Exception as e:
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    try:
        result_df = process_proximity_exposure()
        quick_check_results()
        
    except KeyboardInterrupt:
        print("\n⚠️ Execution interrupted by user")
    except Exception as e:
        import traceback
        traceback.print_exc()
